# Consumer Segmentation Workflow Continuation

This notebook reviews the state of the last workflow run, loads saved artifacts, defines the next analysis steps, executes continuation tasks, and validates the outputs.

## 1. Review Previous Workflow State

Inspect the artifacts and reports generated by the last pipeline execution to determine what work remains.

In [ ]:
from pathlib import Path
import pandas as pd

root = Path('.')
reports_dir = root / 'outputs' / 'reports'
figures_dir = root / 'figures'

print('Reports directory:', reports_dir)
print('Figures directory:', figures_dir)
print('Available report files:')
print(sorted([p.name for p in reports_dir.glob('*.csv')]))


## 2. Load Workflow Checkpoint or Artifacts

Load the processed dataset and evaluation reports so we can continue with interpretation, validation, and delivery outputs.

In [ ]:
segmented = pd.read_csv(reports_dir / 'segmented_data_with_names.csv')
profile = pd.read_csv(reports_dir / 'segment_profiles.csv')
kmeans_eval = pd.read_csv(reports_dir / 'kmeans_evaluation.csv')
gmm_eval = pd.read_csv(reports_dir / 'gmm_evaluation.csv')
stability = pd.read_csv(reports_dir / 'stability_report.csv')

print('Segmented records:', len(segmented))
print('Segment names:', segmented['segment_name'].nunique())
profile.head()


## 3. Define Next Workflow Steps

The next stage includes:
- Reviewing segment profiles and count distributions
- Comparing KMeans and GMM diagnostics for stability and business usefulness
- Saving a concise executive segment summary
- Preparing a final set of charts and CSV outputs for downstream delivery.

In [ ]:
segment_counts = segmented['segment_name'].value_counts().rename_axis('segment_name').reset_index(name='count')
top_segments = segment_counts.sort_values('count', ascending=False)
print(top_segments.to_string(index=False))

print('KMeans silhouette scores:')
print(kmeans_eval[['n_clusters', 'silhouette']].to_string(index=False))

print('GMM BIC scores:')
print(gmm_eval[['n_clusters', 'bic']].to_string(index=False))


## 4. Execute Continuation Tasks

Run the continuation tasks that generate summary diagnostics and save the updated segment outputs.

In [ ]:
summary = pd.DataFrame({
    'segment_name': segment_counts['segment_name'],
    'count': segment_counts['count'],
    'share_pct': segment_counts['count'] / len(segmented) * 100,
})
summary['mean_age'] = summary['segment_name'].map(profile.set_index('segment')['Age'])
summary['mean_income'] = summary['segment_name'].map(profile.set_index('segment')['Annual_Income_(k$)'])
summary['mean_spending_score'] = summary['segment_name'].map(profile.set_index('segment')['Spending_Score'])

summary_path = reports_dir / 'segment_summary_dashboard.csv'
summary.to_csv(summary_path, index=False)
print('Saved segment summary dashboard to', summary_path)
summary


## 5. Validate Outputs and Save Progress

Verify that the newly generated summary file exists and that the segment definitions remain coherent with the previously saved artifacts.

In [ ]:
assert summary_path.exists(), 'Summary file not saved'
assert segmented['segment_name'].notna().all(), 'Missing segment names'
assert profile.shape[0] > 1, 'Segment profile missing'
print('Validation passed. Workflow continuation artifacts are saved.')


## 6. Refine Segment Naming and Interpretations

The original automated names were a useful starting point. In this section we refine naming with a business-oriented label for each cluster and save a more interpretable segment profile.

In [ ]:
refined_names = {
    0: 'Mature Value Seekers',
    1: 'Emerging Mid-Income Explorers',
    2: 'Established Comfort Shoppers',
    3: 'Premium Momentum Spenders',
    4: 'Young High-Interest Savers',
    5: 'Affluent Low-Engagement Professionals',
}
segmented['refined_segment_name'] = segmented['segment'].map(refined_names)
segmented.to_csv(reports_dir / 'segmented_data_with_refined_names.csv', index=False)

count_map = segmented['segment'].value_counts().to_dict()
profile_summary = profile.copy()
profile_summary['segment_name'] = profile_summary['segment'].map(refined_names)
profile_summary['count'] = profile_summary['segment'].map(count_map)
profile_summary['share_pct'] = profile_summary['count'] / len(segmented) * 100
profile_summary = profile_summary.sort_values('share_pct', ascending=False)
profile_summary_path = reports_dir / 'segment_profile_refined.csv'
profile_summary.to_csv(profile_summary_path, index=False)
print('Saved refined segment profile to', profile_summary_path)
profile_summary


## 7. Executive Summary and Strategy Recommendations

Summarize the segmentation findings and translate them into practical marketing actions aligned with business value and segment potential.

In [ ]:
executive_lines = []
executive_lines.append('# Executive Summary')
executive_lines.append('')
executive_lines.append('This segmentation uses KMeans on Age, Annual Income, and Spending Score to identify 6 consumer segments with distinct demographic and behavioral profiles. The solution was selected based on silhouette diagnostics while balancing interpretability and business actionability.')
executive_lines.append('')
executive_lines.append('## Key Findings')
executive_lines.append('')
executive_lines.append('- 6 segments were selected, offering a balance between meaningful differentiation and stable cluster structure. The KMeans silhouette score is highest at k=6, and bootstrap stability testing reports adjusted Rand indices near 0.99 across repeated runs.')
executive_lines.append('- Segments vary by income, spending intent, age, and gender. This enables both value-based targeting and customer lifecycle messaging.')
executive_lines.append('- The largest segment is **Mature Value Seekers**, followed by **Premium Momentum Spenders** and **Balanced/Established middle-income groups**.')
executive_lines.append('')
executive_lines.append('## Recommended Segment Priorities')
executive_lines.append('')
executive_lines.append('- Prioritize **Premium Momentum Spenders** for high-value retention and premium offers. Their high spend and high income indicate strong revenue opportunity.')
executive_lines.append('- Invest in **Affluent Low-Engagement Professionals** with personalized convenience and cross-sell campaigns, as they have strong financial capacity but currently low engagement.')
executive_lines.append('- Cultivate **Emerging Mid-Income Explorers** through discovery campaigns and first-time experience offers, as they are a promising growth segment.')
executive_lines.append('- Maintain efficient retention programs for **Mature Value Seekers** and **Established Comfort Shoppers**; these segments are stable but price- and value-sensitive.')
executive_lines.append('')
executive_lines.append('## Strategic Recommendations')
executive_lines.append('')
executive_lines.append('- Use **income and spending score** signals to personalize promotion intensity: premium goods for high-income/high-spend customers, discounts and loyalty drivers for lower-income/value-oriented customers.')
executive_lines.append('- Align messaging with segment motivations: exclusivity and convenience for high-income segments, affordability and trust for lower-income segments, and engagement opportunities for younger explorers.')
executive_lines.append('- Deploy digital channels for younger, high-interest segments and mix email plus direct outreach for mature value-oriented shoppers.')
executive_lines.append('')
executive_lines.append('## Execution Notes')
executive_lines.append('')
executive_lines.append('- Because this dataset is modest in size and feature scope, further validation with transactional and categorical purchase data is recommended before scaling the segmentation program.')
executive_text = '
'.join(executive_lines)
recommendation_lines = []
recommendation_lines.append('# Segment Recommendations')
recommendation_lines.append('')
recommendation_lines.append('## Premium Momentum Spenders')
recommendation_lines.append('High income, high spending, likely to respond to premium loyalty, VIP experiences, and upselling campaigns.')
recommendation_lines.append('')
recommendation_lines.append('## Affluent Low-Engagement Professionals')
recommendation_lines.append('High income but low spending score. Focus on convenience, time-saving offers, and low-friction purchase triggers.')
recommendation_lines.append('')
recommendation_lines.append('## Emerging Mid-Income Explorers')
recommendation_lines.append('Mid income and medium spend. Invest in trial incentives, social proof, and experience-based messaging.')
recommendation_lines.append('')
recommendation_lines.append('## Young High-Interest Savers')
recommendation_lines.append('Lower income but high spending intent. Offer promotions, bundles, and clear value propositions.')
recommendation_lines.append('')
recommendation_lines.append('## Established Comfort Shoppers')
recommendation_lines.append('Stable middle-income shoppers who value reliability and familiarity. Use retention and convenience messaging.')
recommendation_lines.append('')
recommendation_lines.append('## Mature Value Seekers')
recommendation_lines.append('Older, lower-income shoppers with low spending scores. Maintain trust, loyalty incentives, and price transparency.')
recommendation_text = '
'.join(recommendation_lines)
(reports_dir / 'executive_summary.md').write_text(executive_text, encoding='utf-8')
(reports_dir / 'segment_recommendations.md').write_text(recommendation_text, encoding='utf-8')
print('Saved executive summary and recommendation files to', reports_dir)


In [ ]:
import matplotlib.pyplot as plt
summary = pd.DataFrame({
    'segment_name': segment_counts['segment_name'],
    'count': segment_counts['count'],
    'share_pct': segment_counts['count'] / len(segmented) * 100,
})
summary = summary.sort_values('share_pct', ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(summary['segment_name'], summary['share_pct'], color='tab:blue')
ax.set_title('Segment Share (%) by Refined Name')
ax.set_ylabel('Share (%)')
ax.set_xticklabels(summary['segment_name'], rotation=30, ha='right')
fig.tight_layout()
fig.savefig(figures_dir / 'segment_share.png', dpi=150)
print('Saved segment share figure to', figures_dir / 'segment_share.png')
